In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd

import tzfpy
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import src.makePlots as makePlots

import pickle
from pathlib import Path

import re
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict

import pandas as pd
import plotly.graph_objects as go
from plotly.offline import iplot

import matplotlib.pyplot as plt

from atmoz.resources import plot_utilities, useful_functions, colorbars, default_plot_params

import matplotlib.colors as colors

from collections import defaultdict

import netCDF4 as nc 

def load_pickle(filepath: str): 
    with open(filepath, "rb") as f: 
        temp = pickle.load(f)
    return temp

def dump_pickle(filepath: str, var): 
    with open(filepath, "wb") as f: 
        pickle.dump(var, f)
    return

ModuleNotFoundError: No module named 'src'

In [ ]:
instrument_names = {
    "csl001_hires_guilford": "NOAA CSL Lidar",
    "gsfc003_hires_oldfield": "NASA GSFC Lidar",
    "gsfc003_hires": "NASA GSFC Lidar Glass",
    'csl001_hires_boulder': "NOAA CSL Lidar",
    "york": "CCNY Lidar",
    "larc001_hires_sherwood": "NASA LaRC Lidar"
}

instrument_locations = {
    "csl001_hires_guilford": "Yale Coastal",
    "gsfc003_hires_oldfield": "Flax Pond",
    "gsfc003_hires": "Flax Pond",
    'csl001_hires_boulder': "Boulder, CO",
    "york": "CCNY Rooftop",
    "larc001_hires_sherwood": "LaRC Sherwood"
}

ceilometer_plot_params = {
    "ax.set_ylabel": {
        "ylabel": "Altitude (km ASL)"
        },

    "ax.set_yticks": {
        "ticks": np.arange(0, 11, 1)
        },

    "ax.set_xlabel": {
        "xlabel": "Time [LT]"
        },

    "ax.set_title": {
        "label": r"Attenuated Backscatter Profile",
        "fontsize": 16
        },

    "ax.grid": {
        "visible": True,
        "color": "gray",
        "linestyle": "--",
        "linewidth": 0.5
        },

    "ax.set_ylims": [0, 4],

    "fig.layout": "tight",

    "fig.colorbar": {
        "pad": 0.01, 
        "sub_functions": {
            "set_label": {
                "label": "Attenuated Backscatter (1/m 1/sr)",
                "weight": "bold",
                "size": 16
                },
                    
            "ax.tick_params": {
                "labelsize": 16
                }
            }
        }
    }


def plot_ozone_curtain(start, end, data): 
    for instrument, item in data.items(): 
        lidar = {"X": [], "Y": [], "C": []}
        for filename, df in item.items(): 
            df = df.copy()[start:end]
            if df.empty:
                continue
            
            lidar["X"].append(df.index)
            lidar["Y"].append(df.columns.astype(float) / 1000)
            lidar["C"].append(df.values.T * 1000)
            
        title = f"{instrument}: Vertical Ozone Profile [{start} - {end}]"
        savename = f"{instrument}_{start}_{end}.png".replace(":", "_").replace(" ", "_").replace("-", "_")
        folder = Path("./July26")
        savename = folder / savename
        folder.mkdir(parents=True, exist_ok=True)

        params = {
            "fig.colorbar": {
                "label": "Ozone Mixing Ratio (ppbv)"
                },
            "ax.set_xlabel": {
                "xlabel": "Time [LT]"
                },
            "ax.set_xlim": [pd.Timestamp(start, tz="America/New_York"), pd.Timestamp(end, tz="America/New_York")],
            "ax.set_ylim": [0, 4],
            "ax.set_title": title,
            "fig.savefig": {
                "fname": savename,
                "format": "png",
                "dpi": 600,
                "transparent": True
                },
            "tz": "America/New_York" 
            }

        makePlots.plot_curtain(lidar, **params)  
    return savename

def plot_ceilometer_curtain(start, end, data):
    lidar = {"X": [], "Y": [], "C": []}
    for filename, df in data.items(): 
        df = df.copy()[start:end]
        if df.empty:
            continue

        lidar["X"].append(df.index)
        lidar["Y"].append(df.columns.astype(float) / 1000)
        lidar["C"].append(df.values.T)

    title = f"Flax Pond Luft CHM15K Ceilometer: [{start} to {end}]"

    savename = title.replace(' ', '_').replace('[', '').replace(']', '').replace(',', '').replace('-', '_').replace(":", "_") + ".png"
    folder = Path("./July26")
    savename = folder / savename
    folder.mkdir(parents=True, exist_ok=True)
    
    params = {
        "ax.set_xlim": [pd.Timestamp(start, tz="America/New_York"), pd.Timestamp(end, tz="America/New_York")],
        "ax.set_ylim": [0, 4],
        "ax.set_title": title,
        "fig.savefig": {
            "fname": savename,
            "format": "png",
            "dpi": 600,
            "transparent": True
            },
        }

    cticks=[1e-8, 1e-5]

    norm = colors.LogNorm(vmin=cticks[0], vmax=cticks[1])

    if not isinstance(lidar["X"], list):
        lidar = {key: [lidar[key]] for key in lidar.keys()}    
    
    params = useful_functions.merge_dicts(ceilometer_plot_params, params)
    with plt.rc_context(default_plot_params.curtain_plot_theme):
        fig, ax = plt.subplots()

        for X, Y, C in zip(lidar["X"], lidar["Y"], lidar["C"]):
            im = ax.pcolormesh(X, Y, C, cmap="jet", norm=norm, shading="nearest", alpha=1)

        params["fig.colorbar"]["mappable"] = im

        plot_utilities.apply_datetime_axis(ax, tz="America/New_York")
        
        plot_utilities.apply_plot_params(fig, ax, **params)

        plt.close()
    return savename

In [ ]:
data_dir = Path("../data")

tolnet = load_pickle(data_dir / r"TOLNet.pickle")
filenames = [key for key in tolnet.keys() if "20230725" in key or "20230727" in key or "20230726" in key]

tolnet_data = {}
for filename in filenames:
    instrument_name = instrument_names[filename.split(".")[2]]
    if instrument_name not in tolnet_data:
        tolnet_data[instrument_name] = {}
    tolnet_data[instrument_name][filename] = (tolnet[filename].drop(columns="geometry")
                                              .sort_index()
                                              .sort_index(axis=1)
                                              )
    tolnet_data[instrument_name][filename].index = tolnet_data[instrument_name][filename].index.tz_convert("America/New_York")

### 1. Data Cleaning
- Use an outlier screen process. Look for large non-physical jumps in data from the vertical profile.
- Generate a mask of these values located by height, and time axis

### 2. Removal of Diurnal Cycle
- Time align each curtain / file by time-from-sunrise
- Take the mean, median, and std
- Subtract mean from the curtains 

### 3. Full Vertical Profile
- Take the surface, sonde, TOLNET, Pandora, TEMPO, and GEOS-CF profile and stich it together